# Inferencia reproducible: del diccionario a la predicción

Notebook guiado para clase 1. Evitamos CLI, subprocess y módulos de solución para centrarnos en contratos de entrada, preprocesado e inferencia local reproducible.


## 0. Preparación

Usaremos un modelo scikit-learn pequeño entrenado en la propia libreta con semilla fija. En una práctica posterior el modelo vendrá empaquetado.


In [ ]:
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
wine = load_wine(as_frame=True)
RAW_FEATURES = list(wine.feature_names)
FEATURES = [name.replace("/", "_") for name in RAW_FEATURES]
data = wine.frame.rename(columns={**dict(zip(RAW_FEATURES, FEATURES)), "target": "quality_class"})
TARGET = "quality_class"

display(data.head())
print({"rows": len(data), "features": len(FEATURES), "classes": sorted(data[TARGET].unique())})

## 1. Entrenar un artefacto mínimo para inferir

El entrenamiento es deliberadamente simple. Lo importante es separar entrenamiento, contrato e inferencia.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data[FEATURES], data[TARGET], test_size=0.20, random_state=SEED, stratify=data[TARGET]
)
model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
])
model.fit(X_train, y_train)
print({"test_accuracy": model.score(X_test, y_test)})

## 2. Contrato de entrada con Pydantic

**TODO 1:** revisa los rangos y decide uno que ajustarías para tu dominio. Importa porque el contrato protege al modelo de entradas imposibles. Inspecciona `BaseModel`, `Field` y `extra='forbid'`. Verifica que los casos inválidos lanzan `ValidationError`.


In [ ]:
class WineRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    alcohol: float = Field(ge=0, le=20)
    malic_acid: float = Field(ge=0, le=10)
    ash: float = Field(ge=0, le=5)
    alcalinity_of_ash: float = Field(ge=0, le=40)
    magnesium: float = Field(ge=0, le=200)
    total_phenols: float = Field(ge=0, le=5)
    flavanoids: float = Field(ge=0, le=6)
    nonflavanoid_phenols: float = Field(ge=0, le=1)
    proanthocyanins: float = Field(ge=0, le=5)
    color_intensity: float = Field(ge=0, le=15)
    hue: float = Field(ge=0, le=2)
    od280_od315_of_diluted_wines: float = Field(ge=0, le=5)
    proline: float = Field(ge=0, le=2000)

sample_payload = X_test.iloc[0].to_dict()
request = WineRequest.model_validate(sample_payload)
print(request.model_dump())

## 3. Provocar errores de contrato

**TODO 2:** añade un caso inválido más. Decide qué regla quieres comprobar, porque cada error debe enseñar un riesgo distinto. Inspecciona `ValidationError.errors()`. Verifica que el mensaje señala el campo correcto.


In [ ]:
invalid_cases = {
    "columna extra": {**sample_payload, "unexpected_field": "not allowed"},
    "rango inválido": {**sample_payload, "alcohol": 99},
    # TODO 2: añade otro caso, por ejemplo un tipo no numérico.
}

for name, payload in invalid_cases.items():
    try:
        WineRequest.model_validate(payload)
        print(name, "NO falló")
    except ValidationError as exc:
        print(name, "falló como esperábamos:", exc.errors()[0]["loc"], exc.errors()[0]["msg"])


## 4. Preprocesado: orden explícito de features

El modelo no recibe el diccionario directamente. Recibe una tabla con columnas en el orden esperado.

**TODO 3:** explica por qué no ordenarías columnas alfabéticamente. Importa porque cambiar el orden puede cambiar la predicción. Inspecciona `FEATURES` y verifica el `assert`.


In [ ]:
def preprocess(request: WineRequest) -> pd.DataFrame:
    row = request.model_dump()
    frame = pd.DataFrame([{name: row[name] for name in FEATURES}])
    assert list(frame.columns) == FEATURES
    return frame

features_for_model = preprocess(request)
display(features_for_model)
assert features_for_model.shape == (1, len(FEATURES))

## 5. Función de inferencia

**TODO 4:** decide qué debe devolver la función además de la clase. Importa porque un consumidor necesita entender la confianza y la versión. Inspecciona `predict_proba`. Verifica tipos y rangos con los `assert`.


In [ ]:
def predict_wine(payload: dict) -> dict:
    request = WineRequest.model_validate(payload)
    features = preprocess(request)
    probabilities = model.predict_proba(features)[0]
    class_id = int(np.argmax(probabilities))
    return {
        "predicted_class": class_id,
        "confidence": float(probabilities[class_id]),
        "model_version": "notebook-demo-v1",
        "feature_count": len(FEATURES),
    }

prediction = predict_wine(sample_payload)
print(prediction)
assert isinstance(prediction["predicted_class"], int)
assert 0 <= prediction["confidence"] <= 1

## Cierre

Antes de la clase 2, cada pareja debe poder explicar: qué valida el contrato, dónde vive el orden de columnas, qué devuelve inferencia y qué error mostraría a un consumidor si el contrato falla.
